In [2]:
import hashlib
import time


class Block:
    def __init__(self, height, transactions, previous_hash):
        self.height = height
        self.timestamp = time.time()
        self.transactions = transactions
        self.previous_hash = previous_hash

        self.merkle_root = self.calculate_merkle_root()
        self.hash = self.calculate_hash()


    def calculate_merkle_root(self):
        hashes = [
            hashlib.sha256(tx.encode()).hexdigest()
            for tx in self.transactions
        ]

        if not hashes:
            return hashlib.sha256(b"").hexdigest()

        while len(hashes) > 1:

            if len(hashes) % 2 != 0:
                hashes.append(hashes[-1])

            new_level = []

            for i in range(0, len(hashes), 2):
                combined = hashes[i] + hashes[i + 1]

                new_hash = hashlib.sha256(
                    combined.encode()
                ).hexdigest()

                new_level.append(new_hash)

            hashes = new_level

        return hashes[0]


    def calculate_hash(self):
        block_data = (
            str(self.height)
            + str(self.timestamp)
            + self.previous_hash
            + self.merkle_root
        )

        return hashlib.sha256(
            block_data.encode()
        ).hexdigest()

In [4]:
# Test the Block class

transactions = [
    "Alice sends 10 BTC to Bob",
    "Bob sends 5 BTC to Charlie",
    "Charlie sends 2 BTC to David"
]

block1 = Block(1, transactions, "0")

print("Block Height:", block1.height)
print("Timestamp:", block1.timestamp)
print("Transactions:", block1.transactions)
print("Previous Hash:", block1.previous_hash)
print("Merkle Root:", block1.merkle_root)
print("Block Hash:", block1.hash)

Block Height: 1
Timestamp: 1789449056.1205723
Transactions: ['Alice sends 10 BTC to Bob', 'Bob sends 5 BTC to Charlie', 'Charlie sends 2 BTC to David']
Previous Hash: 0
Merkle Root: 01bf4be56141babf2badbba102129aaaf04179662c40f680a0db0af8d6e28f4e
Block Hash: 2f1b0dce5e96f38188523d4730e2915d5960a4f5ed49141c949128ad84143836


In [3]:
class Blockchain:
    def __init__(self):
        self.chain = [self.create_genesis_block()]

    def create_genesis_block(self):
        return Block(
            height=0,
            transactions=["Genesis Block"],
            previous_hash="0"
        )

    def add_block(self, transactions):
        previous_block = self.chain[-1]

        new_block = Block(
            height=len(self.chain),
            transactions=transactions,
            previous_hash=previous_block.hash
        )

        self.chain.append(new_block)

    def validate_chain(self):
        for i in range(len(self.chain)):

            current_block = self.chain[i]

            if current_block.merkle_root != current_block.calculate_merkle_root():
                return False, i, "Merkle Root mismatch"

            if current_block.hash != current_block.calculate_hash():
                return False, i, "Block hash mismatch"

            if i > 0:
                previous_block = self.chain[i - 1]

                if current_block.previous_hash != previous_block.hash:
                    return False, i, "Previous hash mismatch"

        return True, None, "Blockchain is valid"

In [4]:
blockchain = Blockchain()

In [12]:
blockchain.add_block([
    "Alice sends 1000 BTC to Bob"
])

In [13]:
print("Number of blocks:", len(blockchain.chain))

Number of blocks: 2


In [5]:
# Create a fresh blockchain
blockchain = Blockchain()

# Add 9 blocks with different transaction data
blockchain.add_block(["Alice sends 10 BTC to Bob"])
blockchain.add_block(["Bob sends 5 BTC to Charlie"])
blockchain.add_block(["Charlie sends 2 BTC to David"])
blockchain.add_block(["David sends 3 BTC to Emma"])
blockchain.add_block(["Emma sends 1 BTC to Frank"])
blockchain.add_block(["Frank sends 4 BTC to Grace"])
blockchain.add_block(["Grace sends 2 BTC to Henry"])
blockchain.add_block(["Henry sends 6 BTC to Ivy"])
blockchain.add_block(["Ivy sends 3 BTC to Jack"])

In [6]:
print("========== BLOCKCHAIN BEFORE MODIFICATION ==========")

for block in blockchain.chain:
    print("\nBlock Height:", block.height)
    print("Transactions:", block.transactions)
    print("Previous Hash:", block.previous_hash)
    print("Merkle Root:", block.merkle_root)
    print("Block Hash:", block.hash)

print("\nTotal Blocks:", len(blockchain.chain))

========== BLOCKCHAIN BEFORE MODIFICATION ==========

Block Height: 0
Transactions: ['Genesis Block']
Previous Hash: 0
Merkle Root: 89eb0ac031a63d2421cd05a2fbe41f3ea35f5c3712ca839cbf6b85c4ee07b7a3
Block Hash: bef5e9d8d800803b89d01627e70e5dccf61170e78dfcc715552fa980b36a7f33

Block Height: 1
Transactions: ['Alice sends 10 BTC to Bob']
Previous Hash: bef5e9d8d800803b89d01627e70e5dccf61170e78dfcc715552fa980b36a7f33
Merkle Root: 35f5814fa4c09c39012f1e168a2c94b58fae3000126e158bd0592ff6f5943187
Block Hash: 3650ae946aa66b6ea275afec983cb1ba639b278693f214052762b6f65b5204f3

Block Height: 2
Transactions: ['Bob sends 5 BTC to Charlie']
Previous Hash: 3650ae946aa66b6ea275afec983cb1ba639b278693f214052762b6f65b5204f3
Merkle Root: bfcd81feaa7dc16984efec3e2256d1c47cd21caca8f54de56e9460c4069a698b
Block Hash: 172e7e12fdfd8f217b3cb9e9dd9507e3cd8ccb1a787da03a3ee0a83804a6cfff

Block Height: 3
Transactions: ['Charlie sends 2 BTC to David']
Previous Hash: 172e7e12fdfd8f217b3cb9e9dd9507e3cd8ccb1a787da03a3ee0a8

In [7]:
valid, invalid_block, message = blockchain.validate_chain()

print("\nValidation Result:", valid)
print("Message:", message)


Validation Result: True
Message: Blockchain is valid


In [8]:
valid, invalid_block, message = blockchain.validate_chain()

if valid:
    print("Blockchain is VALID")
else:
    print("Blockchain is INVALID")
    print("First invalid Block Height:", invalid_block)
    print("Reason:", message)

Blockchain is VALID


In [9]:
# Modify transaction data in the 5th block
blockchain.chain[4].transactions[0] = "David sends 300 BTC to Emma"

In [10]:
print("========== MODIFIED 5TH BLOCK ==========")

block = blockchain.chain[4]

print("Block Height:", block.height)
print("Transactions:", block.transactions)
print("Previous Hash:", block.previous_hash)
print("Stored Merkle Root:", block.merkle_root)
print("Stored Block Hash:", block.hash)

========== MODIFIED 5TH BLOCK ==========
Block Height: 4
Transactions: ['David sends 300 BTC to Emma']
Previous Hash: 20a0ac61136f3f3cc2b22a87d28b53ddbe9e29b0528a971f849ebf73f87e96aa
Stored Merkle Root: 9cbea342c58950e8d36ce05545734264a85f650d37115c6dc918219ccd8c958d
Stored Block Hash: ca6877e5afddf46dc0c151ae8397163a8cdb847bc0578f43b95567d8c04cc698


In [11]:
valid, invalid_block, message = blockchain.validate_chain()

print("\n========== VALIDATION AFTER MODIFICATION ==========")

if valid:
    print("Blockchain is VALID")
else:
    print("Blockchain is INVALID")
    print("First invalid Block Height:", invalid_block)
    print("Reason:", message)


========== VALIDATION AFTER MODIFICATION ==========
Blockchain is INVALID
First invalid Block Height: 4
Reason: Merkle Root mismatch


In [12]:
print("Expected Merkle Root:", blockchain.chain[4].merkle_root)
print("Recalculated Merkle Root:", blockchain.chain[4].calculate_merkle_root())

Expected Merkle Root: 9cbea342c58950e8d36ce05545734264a85f650d37115c6dc918219ccd8c958d
Recalculated Merkle Root: 2efe169570e1b54c4f579c90125e285e2e2bbfabeedb922d0bdcbb50740d1c31
